In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os
import sys
import re
from configs.data_model_configs import get_dataset_class
from configs.hparams import get_hparams_class
from models.models import get_backbone_class
from models.models import classifier, CNN_ATTN, codats_classifier
from models.models import HAR_CNN, CNN
import torch.nn as nn
from models.raincoat_modules import tf_encoder, Raincoat_classifier
from models.ssss_tsa_modules import SSSS_SepReps_with_multihead
from models.cluda_modules import CLUDA_Network
import copy

# Add the dataloader directory to the path
sys.path.append('/home/tp2474/AdaTime-adatime_v2/dataloader')
from dataloader import data_generator

def load_checkpoint(checkpoint_path, device):
    """Load model checkpoint"""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    return checkpoint

def extract_target_dataset(experiment_folder_name):
    """
    Extract target dataset name from experiment folder.
    E.g., "mhealth to Pamap2_EXP_HAR" -> "Pamap2"
    """
    match = re.search(r'to\s+([^_\s]+)', experiment_folder_name)    
    if match:
        dataset_name = match.group(1)
        return dataset_name
    return None

def get_data_path(dataset_name):
    """
    Convert dataset name to the corresponding .pkl file path
    """
    base_path = '/home/tp2474/dataset'
    dataset_mapping = {
        'Pamap2': f'{base_path}/Pamap2',
        'RealWorld': f'{base_path}/RealWorld',
        'mhealth': f'{base_path}/mhealth'
    }
    return dataset_mapping.get(dataset_name)

def remap_udar_keys(state_dict):
    """Remap uDAR keys from JoinedNetwork to Sequential format"""
    new_state_dict = {}
    for key, value in state_dict.items():
        if key.startswith('feature_extractor.'):
            new_key = '0.' + key.replace('feature_extractor.', '')
        elif key.startswith('classifier.'):
            new_key = '1.' + key.replace('classifier.', '')
        else:
            new_key = key
        new_state_dict[new_key] = value
    return new_state_dict

class RaincoatSeq(nn.Sequential):
    """tf_encoder returns (features, out_ft); pass only features to the classifier."""
    def forward(self, x):
        f, _ = self[0](x)
        return self[1](f)

class SimpleClassifier(nn.Module):
    """Single-linear-layer classifier (no dense hidden layer).
       Used by EXP3/EXP_NEW checkpoints; EXP_HAR uses models.classifier with dense+logits."""
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.logits = nn.Linear(in_dim, num_classes)
    def forward(self, x):
        return self.logits(x)

def pick_backbone(state_dict):
    """Choose between CNN (conv_block1.0.*) and HAR_CNN (conv1.*) from checkpoint keys."""
    for k in state_dict.keys():
        if "conv_block1" in k:
            return CNN
    return HAR_CNN

def has_dense_classifier(state_dict):
    """True if checkpoint has `1.dense.*` (2-layer classifier); False if just `1.logits.*`."""
    return any(k.startswith("1.dense.") or k == "1.dense.weight" for k in state_dict.keys())


def generate_predictions(model, data_loader, device):
    """Generate predictions for a given data loader"""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            
            outputs = model(batch_x)
            if isinstance(outputs, tuple):
                outputs = outputs[0]
            
            predictions = torch.argmax(outputs, dim=1)
            
            all_preds.append(predictions.cpu().numpy())
            all_labels.append(batch_y.cpu().numpy())
    
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    
    return all_preds, all_labels

def plot_confusion_matrix(cm, class_names, save_path, title='Confusion Matrix'):
    """Plot and save confusion matrix"""
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def main():

    # target-suffix -> list of source-suffixes that should be bundled under it
    TARGETS = {
        "EXP_HAR": ["EXP_HAR"],
        "EXP3":    ["EXP3", "EXP_NEW"],
    }

    experiments_logs_root = '/home/tp2474/AdaTime-adatime_v2/experiments_logs'
    dataset_class = get_dataset_class("yo")
    dataset_configs = dataset_class()
    hparams_class = get_hparams_class("yo")
    hparams_object = hparams_class()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    class_names = ['Sitting', 'Standing', 'Lying', 'Running', 'Walking']

    if not os.path.exists(experiments_logs_root):
        print(f"ERROR: Experiments folder does not exist: {experiments_logs_root}")
        return

    for target_suffix, source_suffixes in TARGETS.items():
        output_dir = os.path.join(experiments_logs_root, 'confusion_matrices', target_suffix)
        os.makedirs(output_dir, exist_ok=True)

        print(f"\n{'#'*70}")
        print(f"TARGET: {target_suffix}  (sources: {source_suffixes})")
        print(f"Output dir: {output_dir}")
        print(f"{'#'*70}")

        all_items = os.listdir(experiments_logs_root)
        experiment_count = 0

        for experiment_folder in all_items:
            experiment_path = os.path.join(experiments_logs_root, experiment_folder)
            if not os.path.isdir(experiment_path) or experiment_folder == 'confusion_matrices':
                continue

            matched_source = next((sfx for sfx in source_suffixes if experiment_folder.endswith(sfx)), None)
            if matched_source is None:
                continue

            # normalized folder name as it would look under the target suffix
            normalized_exp_folder = experiment_folder[:-len(matched_source)] + target_suffix

            experiment_count += 1
            print(f"\n{'='*60}")
            print(f"Processing: {experiment_folder}  ->  {normalized_exp_folder}")
            print(f"{'='*60}")

            target_dataset = extract_target_dataset(experiment_folder)
            if not target_dataset:
                print(f"  ⚠ Could not extract target dataset, skipping...")
                continue

            data_path = get_data_path(target_dataset)
            if not data_path:
                print(f"  ⚠ Unknown dataset '{target_dataset}', skipping...")
                continue

            hparams = {**hparams_object.alg_hparams['Deep_Coral'],
                       **hparams_object.train_params}

            try:
                print(f"    Loading data from: {data_path}")
                data_loaders = data_generator(data_path, dataset_configs, hparams, flag='target')
                test_loader = data_loaders[0]
                print(f"    Test set size: {len(test_loader.dataset)}")
            except Exception as e:
                print(f"    ✗ Error loading data: {e}")
                continue

            print(f"  Target Dataset: {target_dataset}")

            for algorithm_folder in os.listdir(experiment_path):
                algorithm_path = os.path.join(experiment_path, algorithm_folder)
                if not os.path.isdir(algorithm_path):
                    continue

                match = re.match(r'^(.+?)_EXP', algorithm_folder)
                if not match:
                    continue
                da_method = match.group(1)

                print(f"\n  Algorithm: {algorithm_folder}")
                algorithm_output_dir = os.path.join(output_dir, da_method)
                os.makedirs(algorithm_output_dir, exist_ok=True)

                for run_folder in os.listdir(algorithm_path):
                    run_path = os.path.join(algorithm_path, run_folder)
                    if not os.path.isdir(run_path):
                        continue
                    checkpoint_path = os.path.join(run_path, 'checkpoint.pt')
                    if not os.path.exists(checkpoint_path):
                        print(f"    ⚠ No checkpoint in {run_folder}, skipping")
                        continue

                    print(f"    Processing run: {run_folder}")
                    try:
                        checkpoint = load_checkpoint(checkpoint_path, device)
                        state_dict = checkpoint.get('best')
                        if state_dict is None:
                            print(f"      ⚠ No 'best' in checkpoint - skipping")
                            continue

                        # Detect backbone and classifier variant from the checkpoint
                        BB = pick_backbone(state_dict)
                        has_dense = has_dense_classifier(state_dict)
                        def _clf(in_dim):
                            if has_dense:
                                c = copy.copy(dataset_configs)
                                c.final_out_channels = in_dim // dataset_configs.features_len
                                return classifier(c)
                            return SimpleClassifier(in_dim, dataset_configs.num_classes)
                        
                        if 'RAINCOAT' in algorithm_folder:
                            encoder = tf_encoder(dataset_configs).to(device)
                            clf = Raincoat_classifier(dataset_configs).to(device)
                            model = RaincoatSeq(encoder, clf).to(device)
                        elif 'SSSS_TSA' in algorithm_folder:
                            backbone = SSSS_SepReps_with_multihead(dataset_configs, BB).to(device)
                            feat_dim = dataset_configs.final_out_channels * dataset_configs.input_channels * dataset_configs.features_len
                            clf = _clf(feat_dim).to(device)
                            model = nn.Sequential(backbone, clf).to(device)
                        elif 'CLUDA' in algorithm_folder:
                            backbone = CLUDA_Network(BB, dataset_configs).to(device)
                            feat_dim = dataset_configs.final_out_channels * dataset_configs.features_len
                            clf = _clf(feat_dim).to(device)
                            model = nn.Sequential(backbone, clf).to(device)
                        elif 'SASA' in algorithm_folder:
                            sasa_cfg = copy.copy(dataset_configs)
                            sasa_cfg.cnn_blocks = [
                                {'kernel_size': 5, 'maxpool': False, 'dropout': True},
                                {'kernel_size': 8, 'maxpool': False, 'dropout': True},
                                {'kernel_size': 8, 'maxpool': False, 'dropout': True},
                            ]
                            backbone = CNN_ATTN(sasa_cfg).to(device)
                            feat_dim = sasa_cfg.final_out_channels * sasa_cfg.features_len
                            clf = _clf(feat_dim).to(device)
                            model = nn.Sequential(backbone, clf).to(device)
                        elif 'CoDATS' in algorithm_folder:
                            backbone = BB(dataset_configs).to(device)
                            clf = codats_classifier(dataset_configs).to(device)
                            model = nn.Sequential(backbone, clf).to(device)
                        else:
                            backbone = BB(dataset_configs).to(device)
                            feat_dim = dataset_configs.final_out_channels * dataset_configs.features_len
                            clf = _clf(feat_dim).to(device)
                            model = nn.Sequential(backbone, clf).to(device)

                        model.load_state_dict(state_dict)
                        model.eval()
                        model = model.to(device)

                        predictions, true_labels = generate_predictions(model, test_loader, device)
                        cm = confusion_matrix(true_labels, predictions)

                        # filename uses the NORMALIZED experiment folder name (EXP_NEW -> target)
                        filename_base = f"{normalized_exp_folder}_{run_folder}"
                        np.savetxt(os.path.join(algorithm_output_dir, f'{filename_base}_cm.txt'),
                                   cm, fmt='%d')
                        print(f"      ✓ Saved: {algorithm_output_dir}/{filename_base}_cm.txt")

                    except Exception as e:
                        import traceback
                        print(f"      ✗ Error processing {run_folder}: {e}")
                        print(traceback.format_exc())
                        continue

        print(f"\n{'='*60}")
        if experiment_count == 0:
            print(f"WARNING: No experiment folders matched sources {source_suffixes}")
        else:
            print(f"Target {target_suffix}: processed {experiment_count} experiment folders")
        print(f"{'='*60}")

if __name__ == "__main__":
    main()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
import re

GLOBAL_ALIGNMENT_ALGORITHMS = [
    "DDC", "Deep_Coral", "HoMM", "MMDA",
    "DANN", "CoDATS", "AdvSKM",
    "RAINCOAT", "SSSS_TSA"
]

LOCAL_ALIGNMENT_ALGORITHMS = [
    "DSAN", "CDAN", "DIRT", "uDAR", "SWL_Adapt", "MCD",
    "SASA", "CoTMix",
    "DAAN", "ACON", "CLUDA"
]

EXCLUDED_ALGORITHMS = ["NO_ADAPT", "TARGET_ONLY"]

def should_exclude_algorithm(algo_name):
    name_check = algo_name.lower().replace('-', '').replace('_', '')
    for excluded in EXCLUDED_ALGORITHMS:
        if excluded.lower().replace('-', '').replace('_', '') in name_check:
            return True
    return False

def get_algo_sort_key(algo_name):
    name_check = algo_name.lower().replace('-', '').replace('_', '')
    for g in GLOBAL_ALIGNMENT_ALGORITHMS:
        if g.lower().replace('-', '').replace('_', '') in name_check:
            return (0, algo_name)
    for l in LOCAL_ALIGNMENT_ALGORITHMS:
        if l.lower().replace('-', '').replace('_', '') in name_check:
            return (1, algo_name)
    return (2, algo_name)

def format_experiment_name(exp_name, exp_suffix=None):
    """Format task name as 'task <suffix>'. Strips any trailing _EXP* from exp_name."""
    m = re.match(r'^(.+?)_EXP\w*$', exp_name)
    task = m.group(1) if m else exp_name
    return f"{task} {exp_suffix}" if exp_suffix else task

def load_and_aggregate_confusion_matrices(cm_subdirs, max_runs=5):
    """Load and sum cm's across one or more cm/ subfolders, bucketed by experiment name."""
    experiments_data = {}
    for cm_dir in cm_subdirs:
        cm_path = Path(cm_dir)
        if not cm_path.exists():
            continue
        for algo_folder in cm_path.iterdir():
            if not algo_folder.is_dir():
                continue
            algo_name = algo_folder.name
            if should_exclude_algorithm(algo_name):
                continue
            for cm_file in algo_folder.glob("*_cm.txt"):
                try:
                    filename = cm_file.stem
                    run_match = re.search(r'_run_(\d+)_cm$', filename)
                    if run_match and int(run_match.group(1)) >= max_runs:
                        continue
                    # Prefer stripping the EXP-suffix chunk so EXP3 and EXP_NEW share one bucket
                    m_task = re.match(r'^(.+?)_EXP\w*_.+?_run_\d+_cm$', filename)
                    if m_task:
                        experiment_name = m_task.group(1)
                    else:
                        stem_match = re.match(r'^(.+?)_run_\d+_cm$', filename)
                        if not stem_match:
                            continue
                        experiment_name = stem_match.group(1)
                    cm = np.loadtxt(cm_file, dtype=int)
                    experiments_data.setdefault(experiment_name, {}).setdefault(algo_name, []).append(cm)
                except Exception:
                    continue
    return {
        exp: {algo: np.sum(cms, axis=0) for algo, cms in algos.items() if cms}
        for exp, algos in experiments_data.items()
    }

def get_top_misclassifications(algorithm_cms, class_names, top_n=5, num_runs=5):
    all_misclass = {}
    for algo_name, cm in algorithm_cms.items():
        misclass = []
        for true_idx in range(len(class_names)):
            for pred_idx in range(len(class_names)):
                if true_idx != pred_idx:
                    count = cm[true_idx, pred_idx]
                    if count > 0:
                        misclass.append({
                            'True': class_names[true_idx],
                            'Predicted': class_names[pred_idx],
                            'Count': count / num_runs,
                            'Pair': f"{class_names[true_idx]}->{class_names[pred_idx]}"
                        })
        misclass = sorted(misclass, key=lambda x: x['Count'], reverse=True)
        all_misclass[algo_name] = misclass[:top_n]
    return all_misclass

def plot_misclassification_summary(algorithm_cms, class_names, experiment_name,
                                   top_n=5, output_dir=None, num_runs=5, target_suffix=None):
    all_misclass = get_top_misclassifications(algorithm_cms, class_names, top_n, num_runs)
    algo_names = sorted(algorithm_cms.keys(), key=get_algo_sort_key)
    formatted_exp_name = format_experiment_name(experiment_name, target_suffix)
    n_algos = len(algo_names)
    if n_algos == 0:
        return

    n_cols = 3
    n_rows = (n_algos + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
    axes = axes.flatten() if n_algos > 1 else [axes]
    fig.suptitle(f'{formatted_exp_name}\nTop Misclassifications (Averaged over {num_runs} Runs)',
                 fontsize=16, fontweight='bold', y=0.995)

    for idx, algo_name in enumerate(algo_names):
        ax = axes[idx]
        misclass = all_misclass[algo_name]
        group_idx = get_algo_sort_key(algo_name)[0]
        title_color = 'darkblue' if group_idx == 0 else ('darkgreen' if group_idx == 1 else 'black')
        group_label = " (Global)" if group_idx == 0 else (" (Local)" if group_idx == 1 else "")

        if not misclass:
            ax.text(0.5, 0.5, 'No misclassifications', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f"{algo_name}{group_label}", fontsize=12, fontweight='bold', color=title_color)
        else:
            pairs = [m['Pair'] for m in misclass]
            counts = [m['Count'] for m in misclass]
            bars = ax.barh(range(len(pairs)), counts, color='skyblue', edgecolor='black')
            ax.set_yticks(range(len(pairs)))
            ax.set_yticklabels(pairs, fontsize=11)
            ax.set_xlabel('Average Count', fontsize=10)
            ax.set_title(f"{algo_name}{group_label}", fontsize=12, fontweight='bold', color=title_color)
            ax.invert_yaxis()
            ax.grid(axis='x', alpha=0.3)
            for i, (bar, count) in enumerate(zip(bars, counts)):
                ax.text(count, i, f' {count:.1f}', va='center', fontsize=9, fontweight='bold')

    for idx in range(n_algos, len(axes)):
        axes[idx].axis('off')

    plt.tight_layout()
    if output_dir:
        exp_safe_name = formatted_exp_name.replace(' ', '_').replace('/', '_')
        save_path = os.path.join(output_dir, f'{exp_safe_name}_misclassifications_summary.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"    Saved: {save_path}")
    plt.show()
    plt.close()

def plot_individual_confusion_matrix(cm, class_names, algo_name, experiment_name,
                                     output_dir=None, target_suffix=None):
    plt.figure(figsize=(8, 6))
    formatted_exp_name = format_experiment_name(experiment_name, target_suffix)
    cm_normalized = cm / (cm.sum(axis=1, keepdims=True) + 1e-10)
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Proportion'}, vmin=0, vmax=1,
                linewidths=0.5, linecolor='gray')
    plt.title(f'{formatted_exp_name}\n{algo_name}', fontsize=14, fontweight='bold')
    plt.ylabel('True Class', fontsize=11)
    plt.xlabel('Predicted Class', fontsize=11)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    if output_dir:
        safe_exp_name = formatted_exp_name.replace(' ', '_').replace('/', '_')
        safe_algo_name = algo_name.replace(' ', '_').replace('/', '_')
        individual_dir = os.path.join(output_dir, 'individual_cms', safe_exp_name)
        os.makedirs(individual_dir, exist_ok=True)
        save_path = os.path.join(individual_dir, f'{safe_exp_name}_{safe_algo_name}_cm.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_all_confusion_matrices(algorithm_cms, class_names, experiment_name,
                                output_dir=None, num_runs=5, target_suffix=None):
    algo_names = sorted(algorithm_cms.keys(), key=get_algo_sort_key)
    n_algos = len(algo_names)
    if n_algos == 0:
        return None

    n_cols = 3
    n_rows = (n_algos + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
    axes = axes.flatten() if n_algos > 1 else [axes]
    formatted_exp_name = format_experiment_name(experiment_name, target_suffix)
    fig.suptitle(f'{formatted_exp_name}\nConfusion Matrices (Aggregated over {num_runs} Runs)',
                 fontsize=16, fontweight='bold', y=0.995)
    for idx, algo_name in enumerate(algo_names):
        ax = axes[idx]
        cm = algorithm_cms[algo_name]
        total = cm.sum()
        correct = np.trace(cm)
        accuracy = (correct / total) * 100 if total > 0 else 0
        group_idx = get_algo_sort_key(algo_name)[0]
        title_color = 'darkblue' if group_idx == 0 else ('darkgreen' if group_idx == 1 else 'black')
        group_label = " (Global)" if group_idx == 0 else (" (Local)" if group_idx == 1 else "")
        cm_normalized = cm / (cm.sum(axis=1, keepdims=True) + 1e-10)
        sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names,
                    ax=ax, cbar_kws={'label': 'Proportion'},
                    vmin=0, vmax=1, linewidths=0.5, linecolor='gray')
        ax.set_title(f'{algo_name}{group_label}\nAcc: {accuracy:.1f}%',
                     fontsize=11, fontweight='bold', color=title_color)
        ax.set_ylabel('True Class', fontsize=10)
        ax.set_xlabel('Predicted Class', fontsize=10)
        ax.set_xticklabels(class_names, rotation=45, ha='right')
        ax.set_yticklabels(class_names, rotation=0)
    for idx in range(n_algos, len(axes)):
        axes[idx].axis('off')

    plt.tight_layout()
    if output_dir:
        exp_safe_name = formatted_exp_name.replace(' ', '_').replace('/', '_')
        save_path = os.path.join(output_dir, f'{exp_safe_name}_confusion_matrices.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Saved: {save_path}")
    plt.show()
    return fig

def main():
    TARGETS = {
        "EXP_HAR": ["EXP_HAR"],
        "EXP3":    ["EXP3", "EXP_NEW"],
    }
    base_cm = '/home/tp2474/AdaTime-adatime_v2/experiments_logs/confusion_matrices'
    output_root = '/home/tp2474/AdaTime-adatime_v2/experiments_logs/pics'
    class_names = ['Sitting', 'Standing', 'Lying', 'Running', 'Walking']
    NUM_RUNS = 5

    for target_suffix, sub_suffixes in TARGETS.items():
        cm_subdirs = [os.path.join(base_cm, s) for s in sub_suffixes]
        output_dir = os.path.join(output_root, target_suffix)
        os.makedirs(output_dir, exist_ok=True)
        print(f"\n{'#'*70}\nTARGET {target_suffix}\ncm_dirs = {cm_subdirs}\noutput = {output_dir}\n{'#'*70}")

        experiments_data = load_and_aggregate_confusion_matrices(cm_subdirs, max_runs=NUM_RUNS)
        if not experiments_data:
            print(f"No data found for {target_suffix}, skipping.")
            continue

        for exp_name in sorted(experiments_data.keys()):
            algorithm_cms = experiments_data[exp_name]
            formatted_name = format_experiment_name(exp_name, target_suffix)
            print(f"\n{'='*80}\nProcessing: {formatted_name}\n{'='*80}")
            print(f"  Algorithms: {len(algorithm_cms)}")
            for algo_name, cm in algorithm_cms.items():
                plot_individual_confusion_matrix(cm, class_names, algo_name, exp_name,
                                                 output_dir=output_dir, target_suffix=target_suffix)
            plot_misclassification_summary(algorithm_cms, class_names, exp_name,
                                           top_n=5, output_dir=output_dir,
                                           num_runs=NUM_RUNS, target_suffix=target_suffix)
            plot_all_confusion_matrices(algorithm_cms, class_names, exp_name,
                                        output_dir=output_dir,
                                        num_runs=NUM_RUNS, target_suffix=target_suffix)
        print(f"Outputs for {target_suffix} saved to: {output_dir}")

if __name__ == "__main__":
    main()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
import re
import os

GLOBAL_ALIGNMENT_ALGORITHMS = [
    "DDC", "Deep_Coral", "HoMM", "MMDA",
    "DANN", "CoDATS", "AdvSKM",
    "RAINCOAT", "SSSS_TSA"
]

LOCAL_ALIGNMENT_ALGORITHMS = [
    "DSAN", "CDAN", "DIRT", "uDAR", "SWL_Adapt", "MCD",
    "SASA", "CoTMix",
    "DAAN", "ACON", "CLUDA"
]

EXCLUDED_ALGORITHMS = ["NO_ADAPT", "TARGET_ONLY"]

def should_exclude_algorithm(algo_name):
    name_check = algo_name.lower().replace('-', '').replace('_', '')
    for excluded in EXCLUDED_ALGORITHMS:
        if excluded.lower().replace('-', '').replace('_', '') in name_check:
            return True
    return False

def get_algo_sort_key(algo_name):
    name_check = algo_name.lower().replace('-', '').replace('_', '')
    for g in GLOBAL_ALIGNMENT_ALGORITHMS:
        if g.lower().replace('-', '').replace('_', '') in name_check:
            return (0, algo_name)
    for l in LOCAL_ALIGNMENT_ALGORITHMS:
        if l.lower().replace('-', '').replace('_', '') in name_check:
            return (1, algo_name)
    return (2, algo_name)

def algo_group_label(algo_name):
    idx = get_algo_sort_key(algo_name)[0]
    return 'Global' if idx == 0 else ('Local' if idx == 1 else 'Other')

def format_experiment_name(exp_name, exp_suffix=None):
    """Format task name as 'task <suffix>'. Strips any trailing _EXP* from exp_name."""
    m = re.match(r'^(.+?)_EXP\w*$', exp_name)
    task = m.group(1) if m else exp_name
    return f"{task} {exp_suffix}" if exp_suffix else task

def load_and_aggregate_confusion_matrices(cm_subdirs, max_runs=5):
    experiments_data = {}
    for cm_dir in cm_subdirs:
        cm_path = Path(cm_dir)
        if not cm_path.exists():
            continue
        for algo_folder in cm_path.iterdir():
            if not algo_folder.is_dir():
                continue
            algo_name = algo_folder.name
            if should_exclude_algorithm(algo_name):
                continue
            for cm_file in algo_folder.glob("*_cm.txt"):
                try:
                    filename = cm_file.stem
                    run_match = re.search(r'_run_(\d+)_cm$', filename)
                    if run_match and int(run_match.group(1)) >= max_runs:
                        continue
                    # Prefer stripping the EXP-suffix chunk so EXP3 and EXP_NEW share one bucket
                    m_task = re.match(r'^(.+?)_EXP\w*_.+?_run_\d+_cm$', filename)
                    if m_task:
                        experiment_name = m_task.group(1)
                    else:
                        stem_match = re.match(r'^(.+?)_run_\d+_cm$', filename)
                        if not stem_match:
                            continue
                        experiment_name = stem_match.group(1)
                    cm = np.loadtxt(cm_file, dtype=int)
                    experiments_data.setdefault(experiment_name, {}).setdefault(algo_name, []).append(cm)
                except Exception:
                    continue
    return {
        exp: {algo: np.sum(cms, axis=0) for algo, cms in algos.items() if cms}
        for exp, algos in experiments_data.items()
    }

def calculate_group_metrics(cm, class_names):
    indices = {name: i for i, name in enumerate(class_names)}
    groups = {
        'Static (Sit/Stand/Lie)': [indices['Sitting'], indices['Standing'], indices['Lying']],
        'Dynamic (Walk/Run)': [indices['Walking'], indices['Running']],
    }
    metrics = {}
    total = np.sum(cm)
    metrics['Overall Accuracy'] = np.trace(cm) / total if total > 0 else 0.0
    for group_name, group_idxs in groups.items():
        submatrix = cm[np.ix_(group_idxs, group_idxs)]
        inner_correct = np.trace(submatrix)
        inner_total = np.sum(submatrix)
        metrics[f'{group_name} Distinction'] = inner_correct / inner_total if inner_total > 0 else 0.0
    return metrics

def calculate_per_class_f1(cm, class_names):
    per_class = {}
    for i, name in enumerate(class_names):
        tp = cm[i, i]
        fp = np.sum(cm[:, i]) - tp
        fn = np.sum(cm[i, :]) - tp
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        per_class[name] = f1
    return per_class

def plot_metrics_comparison(df, experiment_name, output_dir=None, target_suffix=None):
    df = df.copy()
    df['SortGroup'] = df['Algorithm'].apply(lambda x: get_algo_sort_key(x)[0])
    df = df.sort_values(['SortGroup', 'Algorithm'])
    sorted_algos = df['Algorithm'].unique().tolist()
    plot_df = df.melt(id_vars=['Algorithm', 'SortGroup'], var_name='Metric', value_name='Score')
    plot_df = plot_df[plot_df['Metric'] != 'SortGroup']

    plt.figure(figsize=(15, 8))
    sns.barplot(data=plot_df, x='Metric', y='Score', hue='Algorithm',
                hue_order=sorted_algos, palette='viridis')
    formatted_exp_name = format_experiment_name(experiment_name, target_suffix)
    plt.title(f'{formatted_exp_name}\nClass Awareness & Performance Analysis (Sorted by Group)',
              fontsize=15, fontweight='bold')
    plt.ylim(0, 1.05)
    plt.grid(axis='y', alpha=0.3)
    plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', title='Algorithm\n(Global -> Local)')
    plt.xlabel('')
    plt.tight_layout()
    if output_dir:
        safe_name = formatted_exp_name.replace(' ', '_').replace('/', '_')
        save_path = os.path.join(output_dir, f'{safe_name}_metrics_analysis.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Saved: {save_path}")
    plt.show()
    plt.close()

def main():
    TARGETS = {
        "EXP_HAR": ["EXP_HAR"],
        "EXP3":    ["EXP3", "EXP_NEW"],
    }
    base_cm = '/home/tp2474/AdaTime-adatime_v2/experiments_logs/confusion_matrices'
    output_root = '/home/tp2474/AdaTime-adatime_v2/experiments_logs/pics'
    class_names = ['Sitting', 'Standing', 'Lying', 'Running', 'Walking']
    NUM_RUNS = 5

    for target_suffix, sub_suffixes in TARGETS.items():
        cm_subdirs = [os.path.join(base_cm, s) for s in sub_suffixes]
        output_dir = os.path.join(output_root, target_suffix)
        os.makedirs(output_dir, exist_ok=True)
        print(f"\n{'#'*70}\nTARGET {target_suffix} - class distinction analysis\n{'#'*70}")

        experiments_data = load_and_aggregate_confusion_matrices(cm_subdirs, max_runs=NUM_RUNS)
        if not experiments_data:
            print(f"No data found for {target_suffix}, skipping.")
            continue

        for exp_name in sorted(experiments_data.keys()):
            algorithm_cms = experiments_data[exp_name]
            formatted_name = format_experiment_name(exp_name, target_suffix)
            print(f"\n{'#'*80}\nANALYSIS: {formatted_name}\n{'#'*80}")

            rows, f1_rows = [], []
            for algo_name, cm in algorithm_cms.items():
                rows.append({'Algorithm': algo_name, **calculate_group_metrics(cm, class_names)})
                f1_rows.append({'Algorithm': algo_name, **calculate_per_class_f1(cm, class_names)})
            if not rows:
                continue

            metrics_df = pd.DataFrame(rows)
            metrics_df['Group'] = metrics_df['Algorithm'].apply(algo_group_label)
            metrics_df = metrics_df.sort_values(['Group', 'Overall Accuracy'], ascending=[True, False])

            f1_df = pd.DataFrame(f1_rows)
            f1_df['Group'] = f1_df['Algorithm'].apply(algo_group_label)
            f1_df = f1_df.sort_values(['Group', 'Algorithm']).set_index(['Group', 'Algorithm'])

            print("\nLEADERBOARD (Grouped)")
            print("-" * 80)
            cols = ['Group', 'Algorithm', 'Overall Accuracy'] + [
                c for c in metrics_df.columns
                if c not in ('Group', 'Algorithm', 'Overall Accuracy')
            ]
            print(metrics_df[cols].round(4).to_string(index=False))

            print("\nPER-CLASS F1 SCORES")
            print("-" * 80)
            print(f1_df.round(4).to_string())

            plot_metrics_comparison(metrics_df.drop(columns=['Group']), exp_name,
                                    output_dir=output_dir, target_suffix=target_suffix)

if __name__ == "__main__":
    main()
